In [13]:
import sys
sys.path.append("..")

from pathlib import Path

from utils.token_utils import count_messages_tokens
from utils.prompts import render
from utils.router import pick_model
from utils.llm_client import LLMClient
from utils.config_loader import reload_config
from utils.token_utils import count_messages_tokens

reload_config()

Select model

In [14]:
model = pick_model(
    provider="groq",
    technique="general"
)

print("Model:", model)

llm = LLMClient(
    provider="groq",
    model=model
)

Model: openai/gpt-oss-20b


Define token limit

In [15]:
TOKEN_LIMIT = 150
PROVIDER = "groq"

Create token-count helper

In [16]:
def get_token_count(message, provider, model):

    messages = [
        {
            "role": "user",
            "content": message
        }
    ]

    token_info = count_messages_tokens(
        messages=messages,
        provider=provider,
        model=model
    )

    return token_info["estimated_total"]

Test it:

In [17]:
test_message = "Flood water is rising near our house."

tokens = get_token_count(
    test_message,
    PROVIDER,
    model
)

print("Token count:", tokens)

Token count: 15


Create overflow summarizer

In [18]:
def summarize_overflow(message, llm):

    prompt_text, _ = render(
        "overflow_summarize.v1",

        role="crisis message compression assistant",

        query=message,

        instruction="""
Summarize the message while preserving only information
that is important for crisis-response processing.
""",

        constraints="""
Preserve important information such as:
- location
- number of people or victims
- emergency need
- urgency
- medical information

Remove:
- repeated text
- chain-message content
- unnecessary greetings
- irrelevant commentary
- duplicated information

Do not invent information.
Keep the result concise.
""",

        format="""
Return only the concise processed message.
Do not include explanations or headings.
"""
    )

    response = llm.chat(
        [
            {
                "role": "user",
                "content": prompt_text
            }
        ],
        temperature=0.0,
        max_tokens=100,
        task_type="generation"
    )

    return response["text"].strip()

Create Budget Keeper

In [19]:
def budget_keeper(message, llm, provider="groq"):

    original_tokens = get_token_count(
        message,
        provider,
        llm.model
    )

    print(f"Original token count: {original_tokens}")

    # Message is within budget
    if original_tokens <= TOKEN_LIMIT:

        print("ALLOWED")

        return {
            "status": "ALLOWED",
            "original_tokens": original_tokens,
            "processed_tokens": original_tokens,
            "processed_message": message
        }

    # Message exceeds budget
    print("BLOCKED/TRUNCATED")

    processed_message = summarize_overflow(
        message,
        llm
    )

    processed_tokens = get_token_count(
        processed_message,
        provider,
        llm.model
    )

    return {
        "status": "BLOCKED/TRUNCATED",
        "original_tokens": original_tokens,
        "processed_tokens": processed_tokens,
        "processed_message": processed_message
    }

Test a normal message

In [20]:
normal_message = """
SOS: Three people are trapped in a house in Gampaha.
Flood water is rising and they need a rescue boat.
"""

normal_result = budget_keeper(
    normal_message,
    llm
)

print("\nFinal Result")
print("-" * 50)

print(f"Status: {normal_result['status']}")
print(f"Original tokens: {normal_result['original_tokens']}")
print(f"Processed tokens: {normal_result['processed_tokens']}")

print("\nProcessed Message:")
print(normal_result["processed_message"])

Original token count: 33
ALLOWED

Final Result
--------------------------------------------------
Status: ALLOWED
Original tokens: 33
Processed tokens: 33

Processed Message:

SOS: Three people are trapped in a house in Gampaha.
Flood water is rising and they need a rescue boat.



Create realistic long spam

In [21]:
spam_message = (
    "URGENT PLEASE FORWARD THIS MESSAGE TO EVERYONE. "
    "Please share this with every group immediately. "
    "This message has already been forwarded many times. "
    "Please do not ignore this message and continue sharing. "
) * 10

spam_message += """
There are 3 people trapped on the roof of a house in Gampaha.
Flood water is rising quickly and they need a rescue boat.
One of the victims is an elderly person.
"""

spam_message += (
    "PLEASE FORWARD THIS MESSAGE TO EVERYONE NOW. "
    "Share this message with all contacts and all groups. "
    "Please keep forwarding this message immediately. "
) * 10

In [22]:
spam_tokens = get_token_count(
    spam_message,
    PROVIDER,
    model
)

print("Spam token count:", spam_tokens)

Spam token count: 706


In [23]:
spam_result = budget_keeper(
    spam_message,
    llm
)

print("\nFinal Result")
print("-" * 50)

print(f"Status: {spam_result['status']}")
print(f"Original tokens: {spam_result['original_tokens']}")
print(f"Processed tokens: {spam_result['processed_tokens']}")

print("\nProcessed Message:")
print(spam_result["processed_message"])

Original token count: 706
BLOCKED/TRUNCATED

Final Result
--------------------------------------------------
Status: BLOCKED/TRUNCATED
Original tokens: 706
Processed tokens: 38

Processed Message:
Three people, including an elderly person, are trapped on a roof in Gampaha. Flood water is rising quickly; a rescue boat is urgently needed.


In [25]:
print("\nNORMAL MESSAGE")
print(f"Status           : {normal_result['status']}")
print(f"Original Tokens  : {normal_result['original_tokens']}")
print(f"Processed Tokens : {normal_result['processed_tokens']}")

print("\nSPAM MESSAGE")
print(f"Status           : {spam_result['status']}")
print(f"Original Tokens  : {spam_result['original_tokens']}")
print(f"Processed Tokens : {spam_result['processed_tokens']}")

reduction = (
    spam_result["original_tokens"]
    - spam_result["processed_tokens"]
)

print(f"Token Reduction  : {reduction}")

print("\nProcessed Spam Message:")
print(spam_result["processed_message"])


NORMAL MESSAGE
Status           : ALLOWED
Original Tokens  : 33
Processed Tokens : 33

SPAM MESSAGE
Status           : BLOCKED/TRUNCATED
Original Tokens  : 706
Processed Tokens : 38
Token Reduction  : 668

Processed Spam Message:
Three people, including an elderly person, are trapped on a roof in Gampaha. Flood water is rising quickly; a rescue boat is urgently needed.


Save output

In [26]:
output_path = Path("../output/budget_keeper.md")

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as file:

    file.write("# Part 4 - Budget Keeper\n\n")

    file.write("## Normal Message Test\n\n")
    file.write(
        f"- Status: {normal_result['status']}\n"
    )
    file.write(
        f"- Original Tokens: {normal_result['original_tokens']}\n"
    )
    file.write(
        f"- Processed Tokens: {normal_result['processed_tokens']}\n\n"
    )

    file.write("### Processed Message\n\n")
    file.write(normal_result["processed_message"])
    file.write("\n\n")

    file.write("## Spam Message Test\n\n")
    file.write(
        f"- Status: {spam_result['status']}\n"
    )
    file.write(
        f"- Original Tokens: {spam_result['original_tokens']}\n"
    )
    file.write(
        f"- Processed Tokens: {spam_result['processed_tokens']}\n\n"
    )

    file.write("### Processed Message\n\n")
    file.write(spam_result["processed_message"])
    file.write("\n")

print(f"Saved to: {output_path}")

Saved to: ..\output\budget_keeper.md
